# 01 – Indian Cars Dataset: Exploratory Data Analysis
## 1. Project objective and dataset description
**CarSage AI** will be an Indian car marketplace. This notebook explores the raw data *before* any modelling, to learn whether it can support price prediction, car recommendation and fake-listing detection.

**Rules:** raw CSVs are only read, never modified; no rows are deleted. All cleaning happens on an in-memory copy (`work`).

**Dataset choice (checked against the data, not the filenames):** `cars_ds_final.csv` (1,276 rows) and `cars_ds_final_2021.csv` (1,267 rows) share the same 141 columns. Matching rows on Make+Model+Variant, all 1,250 shared variants have identical prices; the newer file adds 17 variants and fills in tyre/wheel fields that are empty in the 2021 file. So `cars_ds_final.csv` is treated as the more complete version and analysed here. Section 3 re-verifies this.
*Note cells are interpretations; printed outputs are observations.*

## 2. Setup and loading
Paths are set explicitly.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt, seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100); pd.set_option("display.width", 200)

DATA_PATH = Path("../data/raw/archive/cars_ds_final.csv")        # analysed dataset
OTHER_PATH = Path("../data/raw/archive/cars_ds_final_2021.csv")  # older file, used only for comparison
FIG_DIR = Path("../reports/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
MAX_CAT_CHARTS = 15   # cap on categorical charts (the file has 140 columns)

df = pd.read_csv(DATA_PATH)      # read-only; never written back
old = pd.read_csv(OTHER_PATH)
print("Loaded", DATA_PATH.name, df.shape, "| comparison file", OTHER_PATH.name, old.shape)

def save(name):
    plt.tight_layout(); plt.savefig(FIG_DIR / f"{name}.png", dpi=120); plt.show()

## 3. Shape, columns, data types, samples, and comparison with the 2021 file

In [ ]:
print("Rows, columns:", df.shape)
print("dtype counts:", df.dtypes.value_counts().to_dict())
display(df.dtypes.to_frame("dtype").assign(n_unique=df.nunique()).T if False else df.dtypes.to_frame("dtype").assign(n_unique=df.nunique()).head(40))
display(df.iloc[:5, :12]); display(df.sample(5, random_state=0).iloc[:, :12])

In [ ]:
# Compare the two files: same columns? same variants? same prices on shared variants?
key = ["Make", "Model", "Variant"]
def keyed(d):
    d = d.drop(columns="Unnamed: 0").copy()
    d["_k"] = d[key].astype(str).agg("|".join, axis=1); d["_n"] = d.groupby("_k").cumcount()
    return d
m = keyed(old).merge(keyed(df), on=["_k", "_n"], suffixes=("_old", "_new"))
print("Same columns:", list(old.columns) == list(df.columns))
print("Matched variants:", len(m), "| only in new file:", len(df) - len(m), "| only in old file:", len(old) - len(m))
print("Price differences on matched rows:", int((m["Ex-Showroom_Price_old"] != m["Ex-Showroom_Price_new"]).sum()))
diff = {c: int((m[c+"_old"].astype(str) != m[c+"_new"].astype(str)).sum()) for c in old.columns if c != "Unnamed: 0" and c+"_old" in m}
print("Columns with value differences:", {k: v for k, v in diff.items() if v})

## 4. Preparing an analysis copy
Most columns are *text with units* (e.g. price `Rs. 5,06,000`, `624 cc`, `23.6 km/litre`; some mileage values start with `?`). pandas therefore reads them as text. We extract the first number into new `*_num` columns on a **copy** (`work`); the raw text stays untouched.
`Unnamed: 0` is just a row index and is ignored. *Note:* `Power`/`Torque` mix units (PS, bhp, Nm) so their numbers are only roughly comparable.

In [ ]:
work = df.drop(columns="Unnamed: 0").copy()
TEXT_NUMERIC = ["Ex-Showroom_Price", "Displacement", "Fuel_Tank_Capacity", "Height", "Length", "Width", "Wheelbase",
                "Ground_Clearance", "Kerb_Weight", "Gross_Vehicle_Weight", "Boot_Space", "Minimum_Turning_Radius",
                "City_Mileage", "Highway_Mileage", "ARAI_Certified_Mileage", "Power", "Torque", "Electric_Range"]
TEXT_NUMERIC = [c for c in TEXT_NUMERIC if c in work.columns]       # only columns that exist

def first_number(s):
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.extract(r"(\d+\.?\d*)")[0], errors="coerce").where(s.notna())

for c in TEXT_NUMERIC:
    work[c + "_num"] = first_number(work[c])
parsed = pd.DataFrame({"non_null_text": work[TEXT_NUMERIC].notna().sum(),
                       "parsed": work[[c + "_num" for c in TEXT_NUMERIC]].notna().sum().values}, index=TEXT_NUMERIC)
display(parsed)   # parsed should equal non_null_text; a gap means some values didn't contain a number
print("Power units seen:", work["Power"].str.extract(r"(?i)(ps|bhp|kw|hp)")[0].str.lower().value_counts().to_dict())

PRICE = "Ex-Showroom_Price_num"
num_cols = [c for c in work.select_dtypes("number").columns if work[c].notna().sum() > 0]
print("Numeric columns for analysis:", len(num_cols))

## 5. Missing-value analysis
Counts blanks per column on the raw data. Nothing is filled or dropped.

In [ ]:
miss = df.isna().sum().to_frame("missing"); miss["pct"] = (miss.missing / len(df) * 100).round(1)
print("Columns with no missing:", int((miss.missing == 0).sum()), "| >50% missing:", int((miss.pct > 50).sum()), "| >90% missing:", int((miss.pct > 90).sum()))
display(miss.sort_values("pct", ascending=False).head(30))
top = miss[miss.pct > 0].sort_values("pct", ascending=False).head(30).iloc[::-1]
ax = top.pct.plot.barh(figsize=(8, 8)); ax.set_xlabel("% missing"); ax.set_title("Most-missing columns (top 30)")
save("missing_values")

## 6. Duplicate analysis
Checked with and without the `Unnamed: 0` index (which makes every row look unique). Flagged only; nothing removed.

In [ ]:
core = df.drop(columns="Unnamed: 0")
print("Exact duplicate rows (incl. index col):", int(df.duplicated().sum()))
print("Duplicate rows ignoring the index col:", int(core.duplicated().sum()))
print("Duplicate Make+Model+Variant combos:", int(core.duplicated(subset=key, keep=False).sum()), "rows involved")
display(core[core.duplicated(keep=False)].sort_values(key).iloc[:10, :8])

## 7. Descriptive statistics

In [ ]:
display(work[num_cols].describe().T)
base_cat = [c for c in work.select_dtypes("object").columns if not c.endswith("_num")]
display(work[["Make", "Model", "Variant", "Fuel_Type", "Body_Type", "Type", "Drivetrain"]].describe().T if all(c in work for c in ["Make","Model","Variant","Fuel_Type","Body_Type","Type","Drivetrain"]) else work[base_cat[:8]].describe().T)

## 8. Categorical feature distributions
Bar charts for text columns with 2–30 distinct values and <50% missing, plus `Make` (top 20). Capped at a fixed number of charts to keep the 140-column file readable.

In [ ]:
def bar_counts(c, top_n=20):
    vc = work[c].value_counts().head(top_n)
    fig, ax = plt.subplots(figsize=(8, max(2.5, .3 * len(vc))))
    sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax, color="steelblue")
    ax.set_title(f"{c} (top {len(vc)} of {work[c].nunique()}; {work[c].isna().sum()} missing)"); ax.set_xlabel("count")
    save("cat_" + "".join(ch if ch.isalnum() else "_" for ch in c))

cat_ok = [c for c in base_cat if 2 <= work[c].nunique() <= 30 and work[c].isna().mean() < .5 and c not in TEXT_NUMERIC]
cat_ok = ["Fuel_Type", "Body_Type", "Type", "Drivetrain"] + [c for c in cat_ok if c not in ["Fuel_Type", "Body_Type", "Type", "Drivetrain"]]
cat_ok = [c for c in cat_ok if c in work.columns][:MAX_CAT_CHARTS]
for c in (["Make"] if "Make" in work else []) + cat_ok: bar_counts(c)

## 9. Numerical feature distributions
Histogram + box plot per numeric column that is at least 50% populated. Price is also shown on a log scale because prices span lakhs to crores.

In [ ]:
dense = [c for c in num_cols if work[c].notna().mean() >= .5]
print("Plotted (>=50% populated):", dense)
for c in dense:
    s = work[c].dropna()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3))
    sns.histplot(s, kde=s.nunique() > 5, ax=a1); a1.set_title(f"{c}: distribution")
    sns.boxplot(x=s, ax=a2); a2.set_title(f"{c}: box plot")
    save("num_" + "".join(ch if ch.isalnum() else "_" for ch in c))
fig, ax = plt.subplots(figsize=(6, 3)); sns.histplot(np.log10(work[PRICE].dropna()), ax=ax); ax.set_xlabel("log10(price in Rs)"); ax.set_title("Price, log scale")
save("price_log")
print("Price skewness:", round(work[PRICE].skew(), 2), "| after log10:", round(np.log10(work[PRICE]).skew(), 2))

## 10. Outlier investigation
Counts values beyond 1.5×IQR. High-priced cars are legitimately expensive (supercars), so outliers here are *not* automatically errors; inspect before deciding.

In [ ]:
rows = []
for c in dense:
    s = work[c].dropna(); q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1; lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    rows.append({"column": c, "lower": lo, "upper": hi, "n_outliers": int(((s < lo) | (s > hi)).sum()), "min": s.min(), "max": s.max()})
display(pd.DataFrame(rows).set_index("column"))
print("Most expensive variants:"); display(work.nlargest(5, PRICE)[["Make", "Model", "Variant", "Ex-Showroom_Price"]])
print("Cheapest variants:"); display(work.nsmallest(5, PRICE)[["Make", "Model", "Variant", "Ex-Showroom_Price"]])

## 11. Correlation analysis
Spearman correlation (robust to the heavy price skew) among numeric columns that are ≥50% populated, and price by `Make` / `Body_Type`. Correlation is not causation.

In [ ]:
corr = work[dense].corr(method="spearman")
plt.figure(figsize=(9, 7)); sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7}); plt.title("Spearman correlation")
save("correlation_heatmap")
display(corr[PRICE].drop(PRICE).sort_values(key=abs, ascending=False).head(10).to_frame("spearman vs price"))
for c in ["Make", "Body_Type", "Fuel_Type"]:
    if c in work:
        top = work[c].value_counts().head(12).index
        fig, ax = plt.subplots(figsize=(8, 4)); sns.boxplot(data=work[work[c].isin(top)], x=c, y=PRICE, ax=ax); ax.set_yscale("log"); plt.xticks(rotation=60, ha="right"); ax.set_title(f"Price by {c} (log scale)")
        save("price_by_" + c)

## 12. Key findings and limitations
See `reports/eda_summary.md` for the evidence-based summary written from the outputs above.

## 13. Suitability assessment (what the columns allow)
Checks run on the actual column list.

In [ ]:
cols = list(df.columns); low = [c.lower() for c in cols]
hits = lambda words: [c for c, l in zip(cols, low) if any(w in l for w in words)]
print("Price-like columns:", hits(["price"]))
print("Listing/seller/fraud/location/year/odometer-km columns:", hits(["seller", "dealer", "listing", "fraud", "fake", "label", "owner", "city", "year", "location", "url"]) or "none found")
print("Odometer column values:", df["Odometer"].value_counts().to_dict() if "Odometer" in df else "n/a", "(describes the display type, not kilometres driven)")
print("Distinct models:", df["Model"].nunique(), "| distinct makes:", df["Make"].nunique())